In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
# %matplotlib inline
%matplotlib qt5
# import mne
# mne.viz.set_browser_backend("qt")  # or "matplotlib"
# mne.set_config("MNE_BROWSER_BACKEND", "qt")  # or "matplotlib"
%gui qt

import xarray as xr # Assuming you're using this
import numpy as np   # For the example
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import xarray as xr
# import zarr
import panel as pn
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
# import holoviews as hv
# hv.extension('bokeh', logo=False)

# import hvplot.xarray
# import hvplot.pandas
# This line is crucial for displaying plots in a notebook
# hvplot.extension('bokeh') # You can also use 'matplotlib' or 'plotly'

# hv.extension('bokeh')
# hv.extension('matplotlib') # or 'matplotlib'
# hv.extension('plotly') # or 'matplotlib'
# from holoviews import opts
import panel as pn
pn.extension()

import IPython

# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pyphoplacecellanalysis.External.pyqtgraph as pg
from pypho_timeline.widgets import SimpleTimelineWidget, perform_process_all_streams
from pypho_timeline.__main__ import PositionTrackDatasource, VideoTrackDatasource, main, main_all_modalities_from_xdf_file_example

In [ ]:
# demo_xdf_path = Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-21T051157.400Z_eeg.xdf").resolve()
demo_xdf_path = Path(r"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-05T022422.773Z_eeg.xdf").resolve()
assert demo_xdf_path.exists()

In [ ]:
# Create Qt application
app = pg.mkQApp("pyPhoTimelineXDFExample")

timeline = main_all_modalities_from_xdf_file_example(demo_xdf_path)

In [ ]:
# a_track_name: str = 'MOTION_Epoc X Motion'
a_track_name: str = 'EEG_Epoc X'
a_renderer = timeline.track_renderers[a_track_name]
a_detail_renderer = a_renderer.detail_renderer # MotionPlotDetailRenderer 
a_ds = timeline.track_datasources[a_track_name]
# interval = a_ds.intervals_df
interval = a_ds.get_overview_intervals()
# interval = None
type(interval)
interval

In [ ]:
dDisplayItem = timeline.ui.dynamic_docked_widget_container.find_display_dock(identifier=a_track_name) # Dock
a_widget = timeline.ui.matplotlib_view_widgets[a_track_name] # PyqtgraphTimeSynchronizedWidget 
a_root_graphics_layout_widget = a_widget.getRootGraphicsLayoutWidget()
a_plot_item = a_widget.getRootPlotItem()
a_plot_item


In [ ]:
a_ds.detailed_df

In [ ]:
graphics_objects = a_detail_renderer.render_detail(plot_item=a_plot_item, interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
graphics_objects

In [ ]:
# clear_detail
a_detail_renderer.clear_detail(plot_item=a_plot_item, graphics_objects=graphics_objects)
a_detail_renderer

In [ ]:
# a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=a_ds.get_overview_intervals(), detail_data=a_ds.detailed_df) # List[PlotDataItem]
a_bounds_tuple = a_detail_renderer.get_detail_bounds(interval=interval, detail_data=a_ds.detailed_df) # List[PlotDataItem]
print(f'a_bounds_tuple: {a_bounds_tuple}')
x_min, x_max, y_min, y_max = a_bounds_tuple

In [ ]:
# In your notebook, check:
track_name = 'MOTION_Epoc X Motion'
proxy_key = f'track_viewport_{track_name}'
print(f"Connection exists: {proxy_key in timeline.ui.connections}")

In [ ]:
# Check if sigRangeChanged has connections
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    print(f"sigRangeChanged connections: {viewbox.sigRangeChanged.receivers}")

In [ ]:
# Get the current viewport and trigger update
viewbox = a_plot_item.getViewBox()
if viewbox is not None:
    x_range, y_range = viewbox.viewRange()
    a_renderer.update_viewport(x_range[0], x_range[1])

## Individual parts of load from xdf file

In [ ]:
xdf_file_path = demo_xdf_path
print("=" * 60)
print(f"pyPhoTimeline - Load all EEG (or LSL) modalities from XDF: {xdf_file_path}")
print("=" * 60)

# Create Qt application
app = pg.mkQApp("pyPhoTimelineXDFExample")

# --- 1. Load the XDF file (using pyxdf) ---
import pyxdf

print(f"Loading XDF file: {xdf_file_path} ...")
streams, file_header = pyxdf.load_xdf(str(xdf_file_path))
print(f"Streams loaded: {[s['info']['name'][0] for s in streams]}")
print(f"File Header: {file_header}")

In [ ]:
all_streams, all_streams_datasources = perform_process_all_streams(streams=streams)

In [ ]:
all_streams_datasources['Epoc X Motion']


In [ ]:
# active_datasources = eeg_datasources
from pypho_timeline import SynchronizedPlotMode


active_datasources_dict = {k:v for k, v in all_streams_datasources.items() if v is not None}
active_datasources = list(active_datasources_dict.values())


# --- 4. Create the timeline widget and add EEG tracks ---
timeline = SimpleTimelineWidget(
    total_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    total_end_time=max([ds.total_df_start_end_times[1] for ds in active_datasources]),
    window_duration=10.0,
    window_start_time=min([ds.total_df_start_end_times[0] for ds in active_datasources]),
    add_example_tracks=False  # Don't add example tracks for XDF data
)

# Create plot widgets for each EEG stream and add tracks
for datasource in active_datasources:
    # Create a plot widget for this track
    track_widget, root_graphics, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
        name=datasource.custom_datasource_name,
        dockSize=(500, 80),
        dockAddLocationOpts=['bottom'],
        sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
    )
    
    # Set the plot to show the full time range
    plot_item.setXRange(
        timeline.total_data_start_time, 
        timeline.total_data_end_time, 
        padding=0
    )
    plot_item.setYRange(0, 1, padding=0)
    plot_item.setLabel('bottom', 'Time', units='s')
    plot_item.setLabel('left', datasource.custom_datasource_name)
    plot_item.hideAxis('left')  # Hide Y-axis for cleaner look
    
    # Add the track to the plot
    timeline.add_track(
        datasource,
        name=datasource.custom_datasource_name,
        plot_item=plot_item
    )

timeline.setWindowTitle(f"pyPhoTimeline - EEG Modalities from XDF: {xdf_file_path.name}")
timeline.resize(1000, 800)
timeline.show()

print("\nTimeline widget created with EEG tracks from XDF:")
for ds in active_datasources:
    print(f"  - {ds.custom_datasource_name}, time: {ds.total_df_start_end_times}")

print("\nScroll on the timeline to see loaded EEG intervals for each stream.")
print("Close the window to exit.\n")
